# MNIST — Conditional-Free Flow Matching (Straight-Path, Euler & Heun)

Self-contained notebook for `mnist_flow_matching.ipynb`:
- **Auto-Setup**: Automatically loads or computes stratified train/val/test splits, FID reference stats, and evaluation classifier (`auto_setup`).
- **Two Modes**: Set `MODE = 'tune'` (optional automatic 4-trial hyperparameter search) or `MODE = 'train'` (full training + evaluation, using tuned hyperparameters if available or pre-committed defaults).
- **Portable Run Folder**: Everything (checkpoints, curves, sample grids, CSV logs, and `results.json`) is saved inside `runs/{dataset}_{model}_seed{SEED}/` on Google Drive and zipped for easy sharing across Google accounts.

In [ ]:
# 1. Colab setup cell (mounts Google Drive, syncs repo, installs requirements)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/alisamadian/generative-models-benchmark.git'
REPO_DIR = Path('/content/generative-models-benchmark')
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/generative-models-benchmark')

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / 'common').exists():
        os.chdir(DRIVE_REPO_DIR)
    else:
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        else:
            subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--rebase', '--autostash'], check=False)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    if Path.cwd().name == 'notebooks':
        os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Reload common.* modules if already imported in this Python kernel
import importlib
for _mod_name in list(sys.modules.keys()):
    if _mod_name == 'common' or _mod_name.startswith('common.'):
        importlib.reload(sys.modules[_mod_name])

import torch
from common.config import SPLIT_SEED, protocol_hash
from common.utils import library_versions, mount_drive, set_seed

DRIVE_PATHS = mount_drive()
set_seed(SPLIT_SEED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} ({gpu_mem_gb:.2f} GB VRAM)')
else:
    print('GPU: None (running on CPU)')

print('Working directory:', Path.cwd())
print('Library versions:', library_versions())
print('Drive/Cache paths:', {k: str(v) for k, v in DRIVE_PATHS.items()})

## 2. Configuration, Portable Run Folder & Auto-Setup

In [ ]:
DATASET = 'mnist'
MODEL_NAME = 'flow_matching'
MODE = 'train'  # 'tune' (run 4 short validation trials) or 'train' (full training + evaluation)
SEED = 42       # Single training seed (change to any integer if you want another run)

import json
import shutil
from common.config import (
    BATCH_SIZE, CKPT_INTERVAL, EMA_DECAY, SAMPLE_GRID_COUNT, SAMPLE_GRID_SEED,
    TRAIN_STEPS, TUNE_FRACTION, protocol_hash,
)
from common.setup import auto_setup
from common.utils import get_run_dir

RUN_PATHS = get_run_dir(DATASET, MODEL_NAME, seed=SEED, runs_root=DRIVE_PATHS['runs'])
print(f'=== {DATASET.upper()} | {MODEL_NAME.upper()} | MODE={MODE} | SEED={SEED} ===')
print('Portable Run Folder:', RUN_PATHS['run_dir'])
print('Protocol SHA-256 Hash:', protocol_hash())
print('Dataset Train Steps:', TRAIN_STEPS[DATASET], '| Checkpoint Interval:', CKPT_INTERVAL[DATASET])

# Ensure data, stratified train/val/test splits, FID stats, and classifier are ready
SETUP = auto_setup(DATASET, data_root='./data', artifacts_dir='./setup_artifacts')
raw_data = SETUP['raw_data']
split_data = SETUP['split_data']
ref_stats = SETUP['ref_stats']
classifier = SETUP['classifier']

## 3. Data Validation Report (Train & Validation Splits Only; Test Split Locked)

In [ ]:
from common.data import validate_and_report

data_card = validate_and_report(
    dataset=DATASET,
    raw_data=raw_data,
    split_data=split_data,
    include_test=False,
    show_plots=True,
)

## 4. Build Model, Print Architecture Summary & Verify Parameter Count

In [ ]:
from common.nets import build_model, count_parameters
from common.tuning import default_hparams

init_hparams = default_hparams(MODEL_NAME)
model_preview = build_model(DATASET, MODEL_NAME, seed=SEED, hparams=init_hparams)
unet_ref = build_model(DATASET, 'diffusion', seed=SEED)

param_count = count_parameters(model_preview)
unet_param_count = count_parameters(unet_ref)
rel_diff = (param_count - unet_param_count) / float(unet_param_count)

print(f'Model Class: {model_preview.__class__.__name__}')
print(f'Trainable Parameters: {param_count:,} (Reference U-Net: {unet_param_count:,}, diff: {rel_diff:+.1%})')
assert abs(rel_diff) <= 0.25, f'Parameter count {param_count:,} exceeds 25% of U-Net {unet_param_count:,}!'
del model_preview, unet_ref

## 5. Tune Mode (Optional): Run 4 Pre-Committed Validation Trials & Freeze Winner

In [ ]:
from common.training import train_model
from common.tuning import freeze_chosen_hparams, get_tuning_trials
from common.utils import write_results_json

if MODE == 'tune':
    tune_steps = int(round(TUNE_FRACTION * TRAIN_STEPS[DATASET]))
    trials_1_to_3 = get_tuning_trials(MODEL_NAME)[:3]
    scores_1_to_3 = []

    for trial_hp in trials_1_to_3:
        t_idx = int(trial_hp['trial_index'])
        trial_json = RUN_PATHS['tuning'] / f'trial{t_idx}.json'
        legacy_trial_json = Path('results/tuning') / f'{DATASET}_{MODEL_NAME}_trial{t_idx}.json'
        if trial_json.exists():
            saved_t = json.loads(trial_json.read_text(encoding='utf-8'))
            print(f'[Tune Trial {t_idx}] Already completed: val_fid={saved_t["val_fid"]:.2f}')
            scores_1_to_3.append((float(saved_t['val_fid']), float(trial_hp['lr'])))
            continue
        ckpt_dir = RUN_PATHS['tuning'] / f'checkpoints_trial{t_idx}'
        csv_path = RUN_PATHS['logs'] / f'tune_trial{t_idx}.csv'
        tr_out = train_model(
            dataset=DATASET,
            model_name=MODEL_NAME,
            seed=SEED,
            hparams=trial_hp,
            max_steps=tune_steps,
            ckpt_dir=ckpt_dir,
            csv_log_path=csv_path,
            ref_stats_dict=ref_stats,
            raw_data=raw_data,
            split_data=split_data,
            monitor_points=[tune_steps],
        )
        val_fid = tr_out['val_fid_history'][-1]['val_fid']
        val_loss = tr_out['val_loss_history'][-1]['val_loss']
        scores_1_to_3.append((val_fid, float(trial_hp['lr'])))
        t_payload = {
            'run_id': f'{DATASET}_{MODEL_NAME}_tune_trial{t_idx}',
            'dataset': DATASET,
            'model': MODEL_NAME,
            'mode': 'tune',
            'seed': SEED,
            'trial_index': t_idx,
            'hparams': {k: v for k, v in trial_hp.items() if k != 'trial_index'},
            'train_steps': tr_out['final_step'],
            'val_fid': val_fid,
            'val_loss': val_loss,
            'test_touched': False,
        }
        write_results_json(trial_json, t_payload, overwrite_non_final=True)
        write_results_json(legacy_trial_json, t_payload, overwrite_non_final=True)

    best_lr = min(scores_1_to_3, key=lambda x: x[0])[1]
    trial_4_hp = get_tuning_trials(MODEL_NAME, best_lr_from_1_to_3=best_lr)[3]
    trial_4_json = RUN_PATHS['tuning'] / 'trial4.json'
    legacy_trial_4_json = Path('results/tuning') / f'{DATASET}_{MODEL_NAME}_trial4.json'
    if not trial_4_json.exists():
        ckpt_dir_4 = RUN_PATHS['tuning'] / 'checkpoints_trial4'
        csv_path_4 = RUN_PATHS['logs'] / 'tune_trial4.csv'
        tr_out_4 = train_model(
            dataset=DATASET,
            model_name=MODEL_NAME,
            seed=SEED,
            hparams=trial_4_hp,
            max_steps=tune_steps,
            ckpt_dir=ckpt_dir_4,
            csv_log_path=csv_path_4,
            ref_stats_dict=ref_stats,
            raw_data=raw_data,
            split_data=split_data,
            monitor_points=[tune_steps],
        )
        t4_payload = {
            'run_id': f'{DATASET}_{MODEL_NAME}_tune_trial4',
            'dataset': DATASET,
            'model': MODEL_NAME,
            'mode': 'tune',
            'seed': SEED,
            'trial_index': 4,
            'hparams': {k: v for k, v in trial_4_hp.items() if k != 'trial_index'},
            'train_steps': tr_out_4['final_step'],
            'val_fid': tr_out_4['val_fid_history'][-1]['val_fid'],
            'val_loss': tr_out_4['val_loss_history'][-1]['val_loss'],
            'test_touched': False,
        }
        write_results_json(trial_4_json, t4_payload, overwrite_non_final=True)
        write_results_json(legacy_trial_4_json, t4_payload, overwrite_non_final=True)
    else:
        print('[Tune Trial 4] Already completed.')

    chosen_info = freeze_chosen_hparams(
        DATASET,
        MODEL_NAME,
        tuning_dir=RUN_PATHS['tuning'],
        chosen_dir='./results/chosen',
        chosen_path_override=RUN_PATHS['tuning'] / 'chosen.json',
    )
    write_results_json(Path('results/chosen') / f'{DATASET}_{MODEL_NAME}.json', chosen_info, overwrite_non_final=True)
    print('Frozen Chosen Hyperparameters:', json.dumps(chosen_info, indent=2))

## 6. Train Mode: Full Training with Automatic Checkpoint Save & Resume

In [ ]:
from common.training import train_model
from common.tuning import load_chosen_hparams

train_result = None
if MODE in ('train', 'final'):
    # Uses frozen tuning winner if present, or automatically falls back to defaults
    chosen_file = RUN_PATHS['tuning'] / 'chosen.json'
    if not chosen_file.exists() and (Path('results/chosen') / f'{DATASET}_{MODEL_NAME}.json').exists():
        shutil.copy2(Path('results/chosen') / f'{DATASET}_{MODEL_NAME}.json', chosen_file)
    chosen_record = load_chosen_hparams(
        DATASET,
        MODEL_NAME,
        auto_create_default=True,
        chosen_path_override=chosen_file,
    )
    frozen_hparams = chosen_record['hparams']
    print(f'Active Hyperparameters ({chosen_record["selection_reason"]}):', frozen_hparams)

    train_result = train_model(
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        hparams=frozen_hparams,
        max_steps=TRAIN_STEPS[DATASET],
        ckpt_dir=RUN_PATHS['checkpoints'],
        csv_log_path=RUN_PATHS['logs'] / 'training.csv',
        ref_stats_dict=ref_stats,
        raw_data=raw_data,
        split_data=split_data,
    )

## 7. Plot Training Loss, Validation Loss & Validation FID vs Steps

In [ ]:
from common.training import plot_training_curves

if train_result is not None:
    curve_path = RUN_PATHS['figures'] / 'training_curves.png'
    plot_training_curves(
        train_result,
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        save_path=curve_path,
        show_plot=True,
    )
    # Also mirror to ./figures for quick browsing
    Path('figures').mkdir(parents=True, exist_ok=True)
    shutil.copy2(curve_path, Path('figures') / f'curves_{DATASET}_{MODEL_NAME}_seed{SEED}.png')

## 8. Generate Fixed-Seed Sample Grids (seed=777, n=64)

In [ ]:
from common.data import validate_generated_samples
from common.samplers import generate

if train_result is not None:
    ema_model = train_result['ema_model']
    sched = str(frozen_hparams.get('schedule', 'cosine'))
    first_sampler = 'vae' if MODEL_NAME == 'vae' else ('ddim' if MODEL_NAME == 'diffusion' else 'euler')
    nfes_for_grid = [1] if MODEL_NAME == 'vae' else [20, 100]

    for nfe_g in nfes_for_grid:
        grid_samples = generate(
            ema_model,
            sampler=first_sampler,
            nfe=nfe_g,
            n=SAMPLE_GRID_COUNT,
            seed=SAMPLE_GRID_SEED,
            dataset=DATASET,
            schedule=sched,
        )
        grid_fig = RUN_PATHS['figures'] / f'sample_grid_nfe{nfe_g}.png'
        validate_generated_samples(
            grid_samples,
            dataset=DATASET,
            expected_count=SAMPLE_GRID_COUNT,
            show_plot=True,
            save_fig_path=grid_fig,
            title=f'F4: {DATASET.upper()} {MODEL_NAME} (seed={SEED}, sampler={first_sampler}, NFE={nfe_g})',
        )
        shutil.copy2(grid_fig, Path('figures') / f'f4_grid_{DATASET}_{MODEL_NAME}_seed{SEED}_nfe{nfe_g}.png')

## 9. Final Evaluation (Blocks B–H), Save `results.json` & Create Portable `.zip`

In [ ]:
from common.evaluation import evaluate_final
from common.nets import count_parameters
from common.utils import write_results_json

if MODE in ('train', 'final') and train_result is not None:
    metrics_bundle = evaluate_final(
        ema_model=train_result['ema_model'],
        dataset=DATASET,
        model_name=MODEL_NAME,
        seed=SEED,
        ref_stats_dict=ref_stats,
        classifier=classifier,
        schedule=str(frozen_hparams.get('schedule', 'cosine')),
        train_steps=train_result['final_step'],
        train_seconds=train_result['train_seconds'],
        it_per_sec=train_result['it_per_sec'],
        raw_data=raw_data,
        split_data=split_data,
        figures_dir=RUN_PATHS['figures'],
    )

    final_payload = {
        'run_id': f'{DATASET}_{MODEL_NAME}_seed{SEED}',
        'dataset': DATASET,
        'model': MODEL_NAME,
        'mode': 'final',
        'seed': SEED,
        'protocol_hash': protocol_hash(),
        'config_hash': train_result['config_hash'],
        'split_hash': split_data['split_hash'],
        'hparams': frozen_hparams,
        'parameters': count_parameters(train_result['ema_model'], trainable_only=False),
        'train_steps': train_result['final_step'],
        'train_seconds': round(train_result['train_seconds'], 3),
        'it_per_sec': round(train_result['it_per_sec'], 4),
        'ema_decay': EMA_DECAY[DATASET],
        'checkpoint_path': train_result['checkpoint_path'],
        'val_fid_history': train_result['val_fid_history'],
        'data_card': data_card,
        'metrics': metrics_bundle,
        'test_touched': True,
        'rerun_reason': 'updated_run',
    }
    write_results_json(RUN_PATHS['results_json'], final_payload, overwrite_non_final=True)
    write_results_json(Path('results/final') / f'{DATASET}_{MODEL_NAME}_seed{SEED}.json', final_payload, overwrite_non_final=True)

    # Create a portable zip archive of the entire run folder for sharing between Google accounts
    zip_base = DRIVE_PATHS['runs'] / f'{DATASET}_{MODEL_NAME}_seed{SEED}'
    zip_file = shutil.make_archive(str(zip_base), 'zip', root_dir=str(RUN_PATHS['run_dir'].parent), base_dir=RUN_PATHS['run_dir'].name)
    print(f'Saved results JSON to: {RUN_PATHS["results_json"]}')
    print(f'Created portable zip archive at: {zip_file}')